In [63]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno


# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
# from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib
from sklearn import tree 
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from lightgbm import LGBMClassifier 
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid, RadiusNeighborsClassifier

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay

# Custom Functions
from credit_risk_modeling import model_eval
import importlib
importlib.reload(model_eval)

# Class Imbalance
from imblearn.over_sampling import SMOTE

## Imports

In [3]:
X_train = pd.read_csv(
    filepath_or_buffer = "../data/processed/X_train_distance.csv"
)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22686 entries, 0 to 22685
Data columns (total 25 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   numeric__person_age                          22686 non-null  float64
 1   numeric__person_income                       22686 non-null  float64
 2   numeric__person_emp_length                   22686 non-null  float64
 3   numeric__loan_amnt                           22686 non-null  float64
 4   numeric__loan_int_rate                       22686 non-null  float64
 5   numeric__loan_percent_income                 22686 non-null  float64
 6   numeric__cb_person_cred_hist_length          22686 non-null  float64
 7   categorical__person_home_ownership_MORTGAGE  22686 non-null  float64
 8   categorical__person_home_ownership_OTHER     22686 non-null  float64
 9   categorical__person_home_ownership_OWN       22686 non-null  float64
 10

In [4]:
X_train = pd.read_csv(
    filepath_or_buffer = "../data/processed/X_train_distance.csv"
)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22686 entries, 0 to 22685
Data columns (total 25 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   numeric__person_age                          22686 non-null  float64
 1   numeric__person_income                       22686 non-null  float64
 2   numeric__person_emp_length                   22686 non-null  float64
 3   numeric__loan_amnt                           22686 non-null  float64
 4   numeric__loan_int_rate                       22686 non-null  float64
 5   numeric__loan_percent_income                 22686 non-null  float64
 6   numeric__cb_person_cred_hist_length          22686 non-null  float64
 7   categorical__person_home_ownership_MORTGAGE  22686 non-null  float64
 8   categorical__person_home_ownership_OTHER     22686 non-null  float64
 9   categorical__person_home_ownership_OWN       22686 non-null  float64
 10

In [5]:
X_test = pd.read_csv(
    filepath_or_buffer = "../data/processed/X_test_distance.csv"
)
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9723 entries, 0 to 9722
Data columns (total 25 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   numeric__person_age                          9723 non-null   float64
 1   numeric__person_income                       9723 non-null   float64
 2   numeric__person_emp_length                   9723 non-null   float64
 3   numeric__loan_amnt                           9723 non-null   float64
 4   numeric__loan_int_rate                       9723 non-null   float64
 5   numeric__loan_percent_income                 9723 non-null   float64
 6   numeric__cb_person_cred_hist_length          9723 non-null   float64
 7   categorical__person_home_ownership_MORTGAGE  9723 non-null   float64
 8   categorical__person_home_ownership_OTHER     9723 non-null   float64
 9   categorical__person_home_ownership_OWN       9723 non-null   float64
 10  

In [6]:
y_train = pd.read_csv(
    filepath_or_buffer = "../data/interim/y_train.csv"
)
y_train = y_train.values.ravel()

In [7]:
y_test = pd.read_csv(
    filepath_or_buffer = "../data/interim/y_test.csv"
)
y_test.head(1)
y_test = y_test.values.ravel()

## Compare Models

In [ ]:
untuned_models = [
    KNeighborsClassifier(
        algorithm='kd_tree',
        n_jobs=-1
    ),
    RadiusNeighborsClassifier(
        radius=5.0,
        algorithm= 'kd_tree',
        n_jobs=-1
    ),
    NearestCentroid(), # contains internal feature selection
    SVC(
        kernel = 'rbf',
        random_state=42
    ),
]

In [10]:
untuned_model_performances, fitted_models = model_eval.comparing_models(untuned_models, X_train, y_train, X_test, y_test)

In [11]:
untuned_model_performances

,model,roc_auc,pr_auc,log_loss,brier_score,matthews_corrcoef
3,SVC (class_weight=None),0.900937,0.840715,NaN,NaN,0.742034
0,KNeighborsClassifier (n_jobs=-1),0.866809,0.742806,1.331385,0.090816,0.662036
1,RadiusNeighborsClassifier (n_jobs=-1),0.864973,0.700446,0.377223,0.117608,0.442507
2,NearestCentroid,0.847113,0.663724,0.863571,0.184248,0.476991


Low KNN values due to no class imbalance handling

## Hyperparameter Tuning, Class Imbalance, Feature Selection

### SVC

In [196]:
svc = SVC(
    probability= True,
    class_weight='balanced',
    random_state=42
)

In [197]:
svc_param_dist = {
    'C': stats.loguniform(1e-3, 1e3),
    'gamma': stats.loguniform(1e-4, 1e1),
    'kernel': ['poly', 'rbf', 'sigmoid']
}

In [198]:
svc_tuned = HalvingRandomSearchCV(
    estimator=svc,
    param_distributions=svc_param_dist,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

### KNN

In [176]:
knn= KNeighborsClassifier(
    n_jobs=-1
)

In [177]:
knn_param_dist = {
    'n_neighbors': stats.randint(3, 32),
    'weights': ['uniform', 'distance'],
    'algorithm': ['kd_tree', 'ball_tree'],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

In [178]:
knn_tuned = HalvingRandomSearchCV(
    estimator= knn,
    param_distributions= knn_param_dist,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

### Radius Neighbors Classifier

In [201]:
rnc = RadiusNeighborsClassifier(
    radius=5,
    n_jobs=-1
    ),
rnc = rnc[0]

In [202]:
rnc_param_dist = {
    'weights': ['uniform', 'distance'],
    'algorithm': ['kd_tree', 'ball_tree'],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

In [203]:
rnc_tuned = HalvingRandomSearchCV(
    estimator= rnc,
    param_distributions= rnc_param_dist,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [184]:
models = [
    svc_tuned
]

In [185]:
smoted_models = [
    knn_tuned,
    rnc_tuned
]

In [ ]:
tuned_model_performance, fitted_models = model_eval.comparing_manually_tuned_models(
    models,
    X_train,
    y_train,
    X_test,
    y_test,
)

c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan ... nan nan nan]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.75277778 0.81166667 0.77222222]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.7837856  0.78521342 0.79179007]
  warnings.warn(
c:\Users\billy\anaconda3\envs\credit_risk_modeling\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [       nan        nan        nan ... 0.84339753 0.83676668 0.83704842]
  warnings.warn(
c:\Use

In [ ]:
smoted_tuned_performance, fitted_models_smoted = model_eval.comparing_models_smoted(
    smoted_models,
    X_train,
    y_train,
    X_test,
    y_test,
)

In [188]:
tuned_model_performances = pd.concat(
    objs = [tuned_model_performance, smoted_tuned_performance],
    axis=0
)

In [189]:
tuned_model_performances

,model,roc_auc,pr_auc,log_loss,brier_score,matthews_corrcoef,error
0,HalvingRandomSearchCV (n_jobs=-1),0.912942,0.844803,0.258958,0.073840,0.657665,NaN
0,HalvingRandomSearchCV (n_jobs=-1),0.897256,0.815300,0.439204,0.114319,0.582779,NaN
1,HalvingRandomSearchCV (n_jobs=-1),NaN,NaN,NaN,NaN,NaN,No neighbors found for test samples array([ 1...
